## IEC Voter Registration Extraction Approach

Before extracting the data, the IEC Voter Registration Statistics page was inspected to understand how the website delivers its data.

The page does not expose the required municipality and ward registration data directly in the initial HTML. The province, municipality, and ward selections are handled through the IEC's ASP.NET Web Forms interface.

The browser's network activity was therefore inspected to identify what happens when selections are made. This showed that selecting a province triggers an asynchronous POST request to the same IEC page, carrying ASP.NET form-state fields such as `__VIEWSTATE`, `__EVENTVALIDATION`, the selected province, and the event target.

Based on this observation, we will reproduce the website's normal selection process programmatically rather than trying to invent or rely on an undocumented API.

### Extraction flow

`IEC page → KwaZulu-Natal → Municipality → Ward → Registration statistics`

The extraction will:

1. Start an HTTP session with the IEC website.
2. Load the initial page and obtain the required ASP.NET state fields.
3. Select **KwaZulu-Natal** programmatically.
4. Retrieve the available KwaZulu-Natal municipalities.
5. Select each municipality and retrieve its wards.
6. Retrieve the registration statistics available for each ward.
7. Repeat this across the required **2011–2026** period where the IEC source provides the corresponding data.
8. Verify the extracted structure and records.
9. Save the original extracted results to `data/raw/`.

This notebook is limited to **data extraction and verification**. Cleaning, transformation, feature engineering, analysis, and modelling will be handled later in the project.

In [1]:
# We are extracting voter registration statistics
# for KwaZulu-Natal only. from2011 to 2026
#
# The IEC website uses an ASP.NET form, so we will
# reproduce the same province → municipality → ward
# selection process programmatically.
#
# Raw extracted data will be saved in:
# data/raw/
#
# No cleaning or processing is done in this notebook.

from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data directory
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# IEC source
IEC_URL = (
    "https://www.elections.org.za/pw/StatsData/"
    "Voter-Registration-Statistics"
)

# Extraction scope
TARGET_PROVINCE = "KwaZulu-Natal"
KZN_PROVINCE_ID = "4"

# Historical period requested
START_YEAR = 2011
CURRENT_YEAR = 2026

print("IEC extraction setup complete.")
print("Source:", IEC_URL)
print("Province:", TARGET_PROVINCE)
print("Province ID:", KZN_PROVINCE_ID)
print("Period:", START_YEAR, "to", CURRENT_YEAR)
print("Raw directory:", RAW_DIR)

IEC extraction setup complete.
Source: https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics
Province: KwaZulu-Natal
Province ID: 4
Period: 2011 to 2026
Raw directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw


In [2]:
#2.
# We first open the IEC page and keep a session active.
# The session is needed because the IEC uses ASP.NET
# state and cookies when moving between selections.

iec_session = requests.Session()

initial_response = iec_session.get(
    IEC_URL,
    timeout=30
)

print("HTTP status:", initial_response.status_code)
print("Content type:", initial_response.headers.get("Content-Type"))
print("Response size:", len(initial_response.content), "bytes")

assert initial_response.status_code == 200, (
    "IEC voter registration page could not be loaded."
)

initial_soup = BeautifulSoup(
    initial_response.text,
    "html.parser"
)

print("IEC source loaded successfully.")
print("Session established successfully.")

HTTP status: 200
Content type: text/html; charset=utf-8
Response size: 131363 bytes
IEC source loaded successfully.
Session established successfully.


In [3]:
#3.
# The IEC uses ASP.NET Web Forms.
# We need the hidden state fields generated by
# the page before we can reproduce the browser's
# province-selection request.
#
# These values are extracted dynamically because
# they can change between sessions.

form_state = {}

for field in initial_soup.select(
    "input[type='hidden'][name]"
):
    name = field.get("name")
    value = field.get("value", "")

    form_state[name] = value

print("Hidden form fields found:", len(form_state))

required_fields = [
    "__VIEWSTATE",
    "__VIEWSTATEGENERATOR",
    "__EVENTVALIDATION"
]

for field in required_fields:
    assert field in form_state, (
        f"Required ASP.NET field missing: {field}"
    )

print("Required ASP.NET form state found.")

print("\nRequired fields:")
for field in required_fields:
    print("-", field)

Hidden form fields found: 3
Required ASP.NET form state found.

Required fields:
- __VIEWSTATE
- __VIEWSTATEGENERATOR
- __EVENTVALIDATION


In [4]:
#4.
# Lets select the province we want to focus on
# The IEC uses ASP.NET AJAX for the province dropdown.
# Our earlier request reached the server but returned
# ER-500, so we now reproduce the browser request more
# closely, including the AJAX headers.

province_post_data = form_state.copy()

province_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        "ctl00$MainContent$ddlProvinces",

    "__EVENTTARGET":
        "ctl00$MainContent$ddlProvinces",

    "__EVENTARGUMENT":
        "",

    "__LASTFOCUS":
        "",

    "__SCROLLPOSITIONX":
        "0",

    "__SCROLLPOSITIONY":
        "0",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "-1",

    "__ASYNCPOST":
        "true"
})

province_headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "X-MicrosoftAjax": "Delta=true",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": IEC_URL,
    "Origin": "https://www.elections.org.za"
}

province_response = iec_session.post(
    IEC_URL,
    data=province_post_data,
    headers=province_headers,
    timeout=30
)

print("HTTP status:", province_response.status_code)
print("Content type:", province_response.headers.get("Content-Type"))
print("Response size:", len(province_response.content), "bytes")

print("\nResponse preview:")
print(province_response.text[:300])

HTTP status: 200
Content type: text/plain; charset=utf-8
Response size: 129533 bytes

Response preview:
1|#||4|112264|updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padd


In [5]:
#5. 
# ============================================
# CELL 5 — VERIFY KZN PROVINCE RESPONSE
# ============================================
#
# The IEC returned an ASP.NET AJAX updatePanel response.
# We extract the updated HTML and check that the
# municipality dropdown was populated for KwaZulu-Natal.

response_text = province_response.text

assert "updatePanel|MainContent_MainUpdatePanel|" in response_text, (
    "IEC did not return the expected updatePanel response."
)

# Extract the HTML contained in the updatePanel response
panel_marker = "updatePanel|MainContent_MainUpdatePanel|"

panel_start = response_text.find(panel_marker) + len(panel_marker)
panel_html = response_text[panel_start:]

# Parse the returned HTML
province_soup = BeautifulSoup(
    panel_html,
    "html.parser"
)

# Find the municipality dropdown
municipality_select = province_soup.select_one(
    "select[name='ctl00$MainContent$ddlMunicipalities']"
)

assert municipality_select is not None, (
    "Municipality dropdown was not found in the KZN response."
)

# Extract municipality options
municipality_options = []

for option in municipality_select.find_all("option"):
    value = option.get("value", "").strip()
    text = option.get_text(strip=True)

    if value and value != "-1":
        municipality_options.append({
            "municipality_id": value,
            "municipality": text
        })

print("KZN province response verified.")
print("Municipalities found:", len(municipality_options))

print("\nFirst municipalities:")
for municipality in municipality_options[:10]:
    print(
        municipality["municipality_id"],
        "→",
        municipality["municipality"]
    )

KZN province response verified.
Municipalities found: 44

First municipalities:
4005 → ETH - eThekwini
4403 → KZN212 - uMdoni
4404 → KZN213 - uMzumbe
4405 → KZN214 - uMuziwabantu
4407 → KZN216 - Ray Nkonyeni
4409 → KZN221 - uMshwathi
4410 → KZN222 - uMngeni
4411 → KZN223 - Mpofana
4412 → KZN224 - iMpendle
4413 → KZN225 - Msunduzi


In [6]:
#6.
# The KZN AJAX request succeeded, but its response
# does not contain the hidden ASP.NET fields.
#
# We inspect the response structure before constructing
# the municipality request.

response_text = province_response.text

print("Response length:", len(response_text))

print("\n--- RESPONSE CONTROL RECORDS ---")

for marker in [
    "updatePanel|",
    "asyncPostBackControlIDs|",
    "postBackControlIDs|",
    "updatePanelIDs|",
    "panelsToRefreshIDs|",
    "formAction|",
    "pageTitle|",
    "scriptBlock|"
]:
    position = response_text.find(marker)

    if position >= 0:
        print(
            f"\n{marker}"
            f"\n{response_text[position:position + 500]}"
        )
    else:
        print(f"\n{marker} NOT FOUND")

Response length: 129533

--- RESPONSE CONTROL RECORDS ---

updatePanel|
updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padding-top: 0px !important;">
                            <div class="card-body d-flex flex-column" style="padding-bottom: 0px !important;">
                                <h1 class="card-title white_text display-4

asyncPostBackControlIDs|
asyncPostBackControlIDs|||0|postBackControlIDs|||62|updatePanelIDs||tctl00$MainContent$MainUpdatePanel,MainContent_MainUpdatePanel|0|childUpdatePanelIDs|||61|panelsToRefreshIDs||ctl00$MainContent$MainUpdatePanel,MainContent_MainUpdatePanel|2|asyncPostBackTimeout||90|31|formAction||./Voter-Registration-Statistics|29|pageTitle||Voter Registration Statistics|

postBackControlIDs|
postBackControlIDs|||62|updatePanelI

In [7]:
#7.
# The successful province response contains the
# updated municipality dropdown.
#
# We inspect its exact HTML and nearby form controls
# so the next postback matches the IEC page structure.

municipality_select = province_soup.select_one(
    "select[name='ctl00$MainContent$ddlMunicipalities']"
)

assert municipality_select is not None, (
    "Municipality dropdown not found."
)

print("Municipality dropdown found.")
print("\nExact municipality control:")
print(municipality_select.prettify()[:5000])

Municipality dropdown found.

Exact municipality control:
<select class="form-control col" id="MainContent_ddlMunicipalities" name="ctl00$MainContent$ddlMunicipalities" onchange="javascript:setTimeout('__doPostBack(\'ctl00$MainContent$ddlMunicipalities\',\'\')', 0)">
 <option selected="selected" value="-1">
  All municipalities
 </option>
 <option value="4005">
  ETH - eThekwini
 </option>
 <option value="4403">
  KZN212 - uMdoni
 </option>
 <option value="4404">
  KZN213 - uMzumbe
 </option>
 <option value="4405">
  KZN214 - uMuziwabantu
 </option>
 <option value="4407">
  KZN216 - Ray Nkonyeni
 </option>
 <option value="4409">
  KZN221 - uMshwathi
 </option>
 <option value="4410">
  KZN222 - uMngeni
 </option>
 <option value="4411">
  KZN223 - Mpofana
 </option>
 <option value="4412">
  KZN224 - iMpendle
 </option>
 <option value="4413">
  KZN225 - Msunduzi
 </option>
 <option value="4414">
  KZN226 - Mkhambathini
 </option>
 <option value="4415">
  KZN227 - Richmond
 </option>
 <opt

In [8]:
#8.
# We inspect the actual HTML form used by the IEC.
# This confirms the form action and any fields that
# must accompany the municipality postback.

main_form = initial_soup.find("form")

assert main_form is not None, (
    "IEC form was not found."
)

print("Form method:", main_form.get("method"))
print("Form action:", main_form.get("action"))

print("\nForm ID:")
print(main_form.get("id"))

print("\nForm name:")
print(main_form.get("name"))

print("\nForm controls:")
for control in main_form.select(
    "input[name], select[name], textarea[name]"
):
    name = control.get("name")
    value = control.get("value", "")

    if name in [
        "__VIEWSTATE",
        "__VIEWSTATEGENERATOR",
        "__EVENTVALIDATION",
        "ctl00$ctl13",
        "ctl00$MainContent$ddlProvinces",
        "ctl00$MainContent$ddlMunicipalities"
    ]:
        print(
            name,
            "=", 
            value[:100] if isinstance(value, str) else value
        )

Form method: post
Form action: ./Voter-Registration-Statistics

Form ID:
uxForm

Form name:
None

Form controls:
__VIEWSTATE = 8EOPkcCTguCwpoGgrxZatT6qCpTNwl9fBzAerWbapBn/j/me9o30XAH1XCkNP6IKICXlfhV6UijB4CQq+2LNPN3fp7w8dVHKz8dm
__VIEWSTATEGENERATOR = 1852F751
__EVENTVALIDATION = XB4wRAowENU+yDTe10JydjyFwoi+OsoLQjTp9G5PfYhkQGs9NO1WC2xRdvKZc6pTJH1Axyp4zMAZuPvd7zgkFQwcnYhMqkMDbbcu
ctl00$MainContent$ddlProvinces = 
ctl00$MainContent$ddlMunicipalities = 


In [9]:
#9.
# The province request returns an ASP.NET AJAX
# delta response. The updated hidden form fields
# are contained in hiddenField records, so we
# extract those values before selecting a municipality.

import re

# Extract hidden fields returned by the province AJAX response
province_state = {}

hidden_field_pattern = re.compile(
    r"\|hiddenField\|([^|]+)\|([^|]*)"
)

for name, value in hidden_field_pattern.findall(
    province_response.text
):
    province_state[name] = value

print("Updated hidden fields found:", len(province_state))

required_fields = [
    "__VIEWSTATE",
    "__VIEWSTATEGENERATOR",
    "__EVENTVALIDATION"
]

for field in required_fields:
    assert field in province_state, (
        f"Updated ASP.NET field missing: {field}"
    )

print("Updated ASP.NET state extracted successfully.")

# Build municipality request using the UPDATED state
municipality_post_data = province_state.copy()

municipality_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        "ctl00$MainContent$ddlMunicipalities",

    "__EVENTTARGET":
        "ctl00$MainContent$ddlMunicipalities",

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "4005",

    "__ASYNCPOST":
        "true"
})

municipality_response = iec_session.post(
    IEC_URL,
    data=municipality_post_data,
    headers=province_headers,
    timeout=30
)

print("\nMunicipality request:")
print("HTTP status:", municipality_response.status_code)
print("Content type:", municipality_response.headers.get("Content-Type"))
print("Response size:", len(municipality_response.content), "bytes")

print("\nResponse preview:")
print(municipality_response.text[:500])

Updated hidden fields found: 8
Updated ASP.NET state extracted successfully.

Municipality request:
HTTP status: 200
Content type: text/plain; charset=utf-8
Response size: 84122 bytes

Response preview:
1|#||4|63930|updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padding-top: 0px !important;">
                            <div class="card-body d-flex flex-column" style="padding-bottom: 0px !important;">
                                <h1 class="card-title white_t


In [10]:
#10. lets verufy the war table 
# The municipality response should contain the
# registered-voter table for its wards.
#
# We verify the table structure before extracting
# individual ward records.

municipality_soup = BeautifulSoup(
    municipality_response.text,
    "html.parser"
)

# Find the ward table
ward_tables = municipality_soup.find_all("table")

print("Tables found:", len(ward_tables))

assert len(ward_tables) > 0, (
    "No tables found in municipality response."
)

# Look for the table containing the expected ward columns
ward_table = None

for table in ward_tables:
    table_text = table.get_text(" ", strip=True)

    if (
        "Ward" in table_text
        and "Voting districts" in table_text
        and "Registered voters" in table_text
    ):
        ward_table = table
        break

assert ward_table is not None, (
    "Ward registration table not found."
)

print("Ward registration table found.")

# Extract table headers
headers = [
    th.get_text(" ", strip=True)
    for th in ward_table.find_all("th")
]

print("\nTable headers:")
for header in headers:
    print("-", header)

Tables found: 1
Ward registration table found.

Table headers:
- Ward
- Voting districts
- Registered voters


In [11]:
#11.
# now that we have confirmed lets extract wards in page 1 as we did find out that is several pages in wards

# We now extract the ward-level registration
# records from the municipality response.
#
# Page 1 contains the first set of wards returned
# by the IEC for the selected municipality.

ward_rows = []

for row in ward_table.find_all("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["td", "th"])
    ]

    # Keep only rows containing the three expected fields
    if len(cells) == 3 and cells[0].isdigit():
        ward_rows.append(cells)

print("Ward records found on page 1:", len(ward_rows))

assert len(ward_rows) > 0, (
    "No ward records were extracted."
)

print("\nFirst 5 ward records:")

for row in ward_rows[:5]:
    print(row)

Ward records found on page 1: 20

First 5 ward records:
['59500001', '14', '19,964']
['59500002', '21', '21,172']
['59500003', '13', '17,103']
['59500004', '9', '20,987']
['59500005', '5', '14,950']


In [12]:
#12.
#Before touching pagination, let's put these records into a structured raw dataset and verify the values


# We convert the extracted IEC rows into a DataFrame
# while preserving the values returned by the source.
#
# No cleaning or transformation is performed here.

page1_df = pd.DataFrame(
    ward_rows,
    columns=[
        "ward",
        "voting_districts",
        "registered_voters"
    ]
)

print("Rows:", len(page1_df))
print("Columns:", list(page1_df.columns))

print("\nFirst 5 records:")
print(page1_df.head())

Rows: 20
Columns: ['ward', 'voting_districts', 'registered_voters']

First 5 records:
       ward voting_districts registered_voters
0  59500001               14            19,964
1  59500002               21            21,172
2  59500003               13            17,103
3  59500004                9            20,987
4  59500005                5            14,950


In [13]:
#13.
# ============================================
# CELL #13 — INSPECT PAGINATION CONTROLS
# ============================================
#
# The IEC pager is rendered as ASP.NET controls.
# We inspect the actual links/buttons returned by
# the municipality response instead of assuming an ID.

# Find all links and inputs related to the ward pager
pager_elements = municipality_soup.find_all(
    lambda tag: (
        tag.name in ["a", "input"]
        and (
            "WardDataPager" in str(tag)
            or "uxWardDataPager" in str(tag)
        )
    )
)

print("Pager elements found:", len(pager_elements))

assert len(pager_elements) > 0, (
    "IEC ward pagination controls were not found."
)

print("\nPagination controls:\n")

for element in pager_elements:
    print(
        "TAG:", element.name,
        "| TEXT:", element.get_text(" ", strip=True),
        "| NAME:", element.get("name"),
        "| HREF:", element.get("href")
    )

Pager elements found: 7

Pagination controls:

TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl00$ctl00 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl01 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl02 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl03 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl04 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl05 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl02$ctl00 | HREF: None


In [14]:
#14. LETS THEN: reproduce the Page 2 click programmatically.
# We reproduce the IEC pager's Page 2 control.
# The ASP.NET state must come from the current
# municipality response.

# Extract the latest ASP.NET state from the
# municipality AJAX response.
municipality_state = {}

for name, value in re.findall(
    r"\|hiddenField\|([^|]+)\|([^|]*)",
    municipality_response.text
):
    municipality_state[name] = value

print("Updated hidden fields found:", len(municipality_state))

assert "__VIEWSTATE" in municipality_state
assert "__EVENTVALIDATION" in municipality_state

# Page 2 control identified from the actual IEC response
page2_control = (
    "ctl00$MainContent$uxRegisteredVotersWardListView"
    "$uxWardDataPager$ctl01$ctl02"
)

page2_post_data = municipality_state.copy()

page2_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        + page2_control,

    "__EVENTTARGET": page2_control,

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "4005",

    "__ASYNCPOST":
        "true"
})

page2_response = iec_session.post(
    IEC_URL,
    data=page2_post_data,
    headers=province_headers,
    timeout=30
)

print("\nPage 2 request:")
print("HTTP status:", page2_response.status_code)
print("Content type:", page2_response.headers.get("Content-Type"))
print("Response size:", len(page2_response.content), "bytes")

print("\nResponse preview:")
print(page2_response.text[:500])

Updated hidden fields found: 8

Page 2 request:
HTTP status: 200
Content type: text/plain; charset=utf-8
Response size: 84083 bytes

Response preview:
1|#||4|63891|updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padding-top: 0px !important;">
                            <div class="card-body d-flex flex-column" style="padding-bottom: 0px !important;">
                                <h1 class="card-title white_t


In [15]:
#15.
# We parse the Page 2 response and confirm that
# it contains ward records different from Page 1.

page2_soup = BeautifulSoup(
    page2_response.text,
    "html.parser"
)

# Find the ward table
page2_ward_table = None

for table in page2_soup.find_all("table"):
    table_text = table.get_text(" ", strip=True)

    if (
        "Ward" in table_text
        and "Voting districts" in table_text
        and "Registered voters" in table_text
    ):
        page2_ward_table = table
        break

assert page2_ward_table is not None, (
    "Ward registration table not found on Page 2."
)

# Extract Page 2 ward records
page2_rows = []

for row in page2_ward_table.find_all("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["td", "th"])
    ]

    if len(cells) == 3 and cells[0].isdigit():
        page2_rows.append(cells)

print("Page 2 ward records:", len(page2_rows))

assert len(page2_rows) > 0, (
    "No ward records found on Page 2."
)

print("\nFirst 5 Page 2 records:")

for row in page2_rows[:5]:
    print(row)

# Confirm Page 2 is different from Page 1
page1_wards = {row[0] for row in ward_rows}
page2_wards = {row[0] for row in page2_rows}

print("\nPage 1 first ward:", ward_rows[0][0])
print("Page 2 first ward:", page2_rows[0][0])

assert page1_wards.isdisjoint(page2_wards), (
    "Page 2 contains wards already found on Page 1."
)

print("\nPage 2 verification passed.")

Page 2 ward records: 20

First 5 Page 2 records:
['59500041', '5', '17,053']
['59500042', '8', '18,545']
['59500043', '7', '16,185']
['59500044', '7', '16,686']
['59500045', '7', '17,493']

Page 1 first ward: 59500001
Page 2 first ward: 59500041

Page 2 verification passed.


In [16]:
#16. now lets extract page 2 into a Dataframe,

# We convert the verified Page 2 records into
# the same raw structure used for Page 1.

page2_df = pd.DataFrame(
    page2_rows,
    columns=[
        "ward",
        "voting_districts",
        "registered_voters"
    ]
)

print("Rows:", len(page2_df))
print("Columns:", list(page2_df.columns))

print("\nFirst 5 Page 2 records:")
print(page2_df.head())


Rows: 20
Columns: ['ward', 'voting_districts', 'registered_voters']

First 5 Page 2 records:
       ward voting_districts registered_voters
0  59500041                5            17,053
1  59500042                8            18,545
2  59500043                7            16,185
3  59500044                7            16,686
4  59500045                7            17,493


In [17]:
#17.
# Pages 1 and 2 were verified manually.
# We now repeat the same ASP.NET pagination process
# for Pages 3, 4, and 5.

additional_page_rows = {}

for page_number in range(3, 6):

    print(f"\nRequesting Page {page_number}...")

    # Extract the latest ASP.NET state from the
    # response of the previous page.
    current_state = {}

    for name, value in re.findall(
        r"\|hiddenField\|([^|]+)\|([^|]*)",
        page2_response.text
        if page_number == 3
        else current_page_response.text
    ):
        current_state[name] = value

    assert "__VIEWSTATE" in current_state
    assert "__EVENTVALIDATION" in current_state

    # Page controls:
    # Page 3 = ctl01$ctl03
    # Page 4 = ctl01$ctl04
    # Page 5 = ctl01$ctl05
    page_control = (
        "ctl00$MainContent$uxRegisteredVotersWardListView"
        "$uxWardDataPager$ctl01$ctl0"
        + str(page_number)
    )

    page_post_data = current_state.copy()

    page_post_data.update({
        "ctl00$ctl13":
            "ctl00$MainContent$MainUpdatePanel|"
            + page_control,

        "__EVENTTARGET": page_control,
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": "",

        "ctl00$MainContent$ddlProvinces":
            KZN_PROVINCE_ID,

        "ctl00$MainContent$ddlMunicipalities":
            "4005",

        "__ASYNCPOST":
            "true"
    })

    current_page_response = iec_session.post(
        IEC_URL,
        data=page_post_data,
        headers=province_headers,
        timeout=30
    )

    print(
        "HTTP status:",
        current_page_response.status_code
    )

    print(
        "Response size:",
        len(current_page_response.content),
        "bytes"
    )

    assert current_page_response.status_code == 200
    assert "updatePanel|MainContent_MainUpdatePanel" in (
        current_page_response.text
    )

    # Parse the page
    current_soup = BeautifulSoup(
        current_page_response.text,
        "html.parser"
    )

    current_ward_table = None

    for table in current_soup.find_all("table"):
        table_text = table.get_text(" ", strip=True)

        if (
            "Ward" in table_text
            and "Voting districts" in table_text
            and "Registered voters" in table_text
        ):
            current_ward_table = table
            break

    assert current_ward_table is not None, (
        f"Ward table not found on Page {page_number}."
    )

    # Extract ward rows
    rows = []

    for row in current_ward_table.find_all("tr"):
        cells = [
            cell.get_text(" ", strip=True)
            for cell in row.find_all(["td", "th"])
        ]

        if len(cells) == 3 and cells[0].isdigit():
            rows.append(cells)

    print(
        f"Page {page_number} ward records:",
        len(rows)
    )

    assert len(rows) == 20, (
        f"Expected 20 wards on Page {page_number}, "
        f"found {len(rows)}."
    )

    additional_page_rows[page_number] = rows

print("\nPages 3–5 extraction completed successfully.")


Requesting Page 3...
HTTP status: 200
Response size: 84083 bytes
Page 3 ward records: 20

Requesting Page 4...
HTTP status: 200
Response size: 84085 bytes
Page 4 ward records: 20

Requesting Page 5...
HTTP status: 200
Response size: 72975 bytes
Page 5 ward records: 12


AssertionError: Expected 20 wards on Page 5, found 12.

In [18]:
#18.
#lets investigate and be sure by what is returned by page 5
# The final IEC page may contain fewer records
# than the standard page size.
#
# Page 5 returned 12 wards, so we verify that
# these are valid ward records rather than treating
# the smaller final page as an extraction failure.

page5_rows = rows

print("Page 5 ward records:", len(page5_rows))

assert len(page5_rows) > 0, (
    "No ward records were extracted from Page 5."
)

print("\nPage 5 records:")

for row in page5_rows:
    print(row)

print("\nPage 5 verification passed.")

Page 5 ward records: 12

Page 5 records:
['59500101', '4', '18,890']
['59500102', '5', '16,632']
['59500103', '12', '21,012']
['59500104', '4', '15,719']
['59500105', '26', '16,780']
['59500106', '6', '16,132']
['59500107', '5', '16,071']
['59500108', '10', '15,302']
['59500109', '7', '19,849']
['59500110', '5', '19,858']
['59500111', '11', '15,638']
['59500112', '6', '17,140']

Page 5 verification passed.


In [19]:
#19. 
#We combine the verified ward pages into one
# raw municipality-level dataset.
#
# No cleaning or transformation is performed.
# The values remain exactly as returned by the IEC.

all_ethekwini_rows = (
    ward_rows
    + page2_rows
    + additional_page_rows[3]
    + additional_page_rows[4]
    + page5_rows
)

ethekwini_df = pd.DataFrame(
    all_ethekwini_rows,
    columns=[
        "ward",
        "voting_districts",
        "registered_voters"
    ]
)

print("Total ward records:", len(ethekwini_df))

print(
    "Unique ward records:",
    ethekwini_df["ward"].nunique()
)

print("\nColumns:")
print(list(ethekwini_df.columns))

assert len(ethekwini_df) == 92, (
    f"Expected 92 ward records, "
    f"found {len(ethekwini_df)}."
)

assert ethekwini_df["ward"].nunique() == 92, (
    "Duplicate ward records detected."
)

print("\nFirst 5 wards:")
print(ethekwini_df.head())

print("\nLast 5 wards:")
print(ethekwini_df.tail())

print("\nEThekwini ward extraction verification passed.")

Total ward records: 92
Unique ward records: 92

Columns:
['ward', 'voting_districts', 'registered_voters']

First 5 wards:
       ward voting_districts registered_voters
0  59500001               14            19,964
1  59500002               21            21,172
2  59500003               13            17,103
3  59500004                9            20,987
4  59500005                5            14,950

Last 5 wards:
        ward voting_districts registered_voters
87  59500108               10            15,302
88  59500109                7            19,849
89  59500110                5            19,858
90  59500111               11            15,638
91  59500112                6            17,140

EThekwini ward extraction verification passed.


In [20]:
#20. 
#okay we have executed a strategy test and veruified it so
# Before we automate the other municipalities, let's save this verified extraction.
#This follows our EXTRACT then VERIFY then SAVE RAW workflow. and what is nice no duplicates detected the logic seems to be promising,

# The eThekwini ward extraction has been verified,
# so we now save the raw IEC response data.


ethekwini_raw_path = (
    RAW_DIR / "iec_kzn_ethekwini_ward_registration_2026.csv"
)

ethekwini_df.to_csv(
    ethekwini_raw_path,
    index=False
)

print("Raw file saved:")
print(ethekwini_raw_path)

print("\nRows saved:", len(ethekwini_df))
print("Columns saved:", list(ethekwini_df.columns))

# Verify that the saved file can be read back
saved_ethekwini_df = pd.read_csv(
    ethekwini_raw_path,
    dtype=str
)

print("\nSaved file verification:")
print("Rows read back:", len(saved_ethekwini_df))
print("Columns read back:", list(saved_ethekwini_df.columns))

assert len(saved_ethekwini_df) == 92
assert list(saved_ethekwini_df.columns) == [
    "ward",
    "voting_districts",
    "registered_voters"
]

print("\nEThekwini raw file verification passed.")

Raw file saved:
C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw\iec_kzn_ethekwini_ward_registration_2026.csv

Rows saved: 92
Columns saved: ['ward', 'voting_districts', 'registered_voters']

Saved file verification:
Rows read back: 92
Columns read back: ['ward', 'voting_districts', 'registered_voters']

EThekwini raw file verification passed.


In [21]:
#21.
# so in moving away from hard-coding 4005 (eThekwini) and use the 44 municipalities already returned by the KZN province response.

#This cell only inspects and verifies that municipality list. It does not extract the municipalities yet. we working towards applying the above confirmed logic

# The KZN province response returned the municipalities
# available under KwaZulu-Natal.
#
# We now structure that list so the remaining extraction
# can iterate through every KZN municipality.
#
# No ward data is extracted in this cell.

municipality_options = province_soup.select(
    "select[name='ctl00$MainContent$ddlMunicipalities'] option"
)

kzn_municipalities = []

for option in municipality_options:
    municipality_id = option.get("value", "").strip()
    municipality_name = option.get_text(" ", strip=True)

    if municipality_id and municipality_id != "-1":
        kzn_municipalities.append({
            "municipality_id": municipality_id,
            "municipality": municipality_name
        })

kzn_municipalities_df = pd.DataFrame(
    kzn_municipalities
)

print("KZN municipalities found:", len(kzn_municipalities_df))

print("\nColumns:")
print(list(kzn_municipalities_df.columns))

print("\nFirst 10 municipalities:")
print(kzn_municipalities_df.head(10).to_string(index=False))

assert len(kzn_municipalities_df) == 44, (
    f"Expected 44 KZN municipalities, "
    f"found {len(kzn_municipalities_df)}."
)

assert kzn_municipalities_df[
    "municipality_id"
].nunique() == 44, (
    "Duplicate municipality IDs detected."
)

print("\nKZN municipality list verification passed.")


KZN municipalities found: 44

Columns:
['municipality_id', 'municipality']

First 10 municipalities:
municipality_id          municipality
           4005       ETH - eThekwini
           4403       KZN212 - uMdoni
           4404      KZN213 - uMzumbe
           4405 KZN214 - uMuziwabantu
           4407 KZN216 - Ray Nkonyeni
           4409    KZN221 - uMshwathi
           4410      KZN222 - uMngeni
           4411      KZN223 - Mpofana
           4412     KZN224 - iMpendle
           4413     KZN225 - Msunduzi

KZN municipality list verification passed.


In [22]:
#22. lets test the next municiplaity to be sure:


# We have verified eThekwini successfully.
# We now test the same municipality-selection
# process on uMdoni before automating all 44
# KZN municipalities.
#
# This helps detect municipality-specific
# pagination or response differences early.

test_municipality_id = "4403"
test_municipality_name = "KZN212 - uMdoni"

print("Testing municipality:")
print("ID:", test_municipality_id)
print("Name:", test_municipality_name)

# Use the latest state from the KZN province response
test_municipality_state = {}

for name, value in re.findall(
    r"\|hiddenField\|([^|]+)\|([^|]*)",
    province_response.text
):
    test_municipality_state[name] = value

assert "__VIEWSTATE" in test_municipality_state
assert "__EVENTVALIDATION" in test_municipality_state

test_municipality_post_data = (
    test_municipality_state.copy()
)

test_municipality_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        "ctl00$MainContent$ddlMunicipalities",

    "__EVENTTARGET":
        "ctl00$MainContent$ddlMunicipalities",

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        test_municipality_id,

    "__ASYNCPOST":
        "true"
})

test_municipality_response = iec_session.post(
    IEC_URL,
    data=test_municipality_post_data,
    headers=province_headers,
    timeout=30
)

print("\nResponse:")
print("HTTP status:", test_municipality_response.status_code)
print(
    "Response size:",
    len(test_municipality_response.content),
    "bytes"
)

assert test_municipality_response.status_code == 200
assert "updatePanel|MainContent_MainUpdatePanel" in (
    test_municipality_response.text
)

print("\nMunicipality response received successfully.")

Testing municipality:
ID: 4403
Name: KZN212 - uMdoni

Response:
HTTP status: 200
Response size: 81860 bytes

Municipality response received successfully.


In [23]:
#23. lets inspect the wards in this municipality:

# We verify that uMdoni returns the expected
# IEC ward registration table.
#
# No extraction or transformation beyond
# reading the table structure is done here.

test_municipality_soup = BeautifulSoup(
    test_municipality_response.text,
    "html.parser"
)

test_ward_tables = test_municipality_soup.find_all("table")

print("Tables found:", len(test_ward_tables))

assert len(test_ward_tables) > 0, (
    "No tables found in uMdoni response."
)

test_ward_table = None

for table in test_ward_tables:
    table_text = table.get_text(" ", strip=True)

    if (
        "Ward" in table_text
        and "Voting districts" in table_text
        and "Registered voters" in table_text
    ):
        test_ward_table = table
        break

assert test_ward_table is not None, (
    "uMdoni ward registration table not found."
)

print("uMdoni ward registration table found.")

test_headers = [
    th.get_text(" ", strip=True)
    for th in test_ward_table.find_all("th")
]

print("\nTable headers:")
for header in test_headers:
    print("-", header)

assert test_headers == [
    "Ward",
    "Voting districts",
    "Registered voters"
]

print("\nuMdoni ward table verification passed.")

Tables found: 1
uMdoni ward registration table found.

Table headers:
- Ward
- Voting districts
- Registered voters

uMdoni ward table verification passed.


In [24]:
#24. 
# now we need to determine how many pages uMdoni has rather than assuming five pages like eThekwini. byu inspecting its pagination


# Municipalities can have different numbers of wards,
# so we must detect pagination dynamically rather than
# assuming every municipality has five pages.

test_pager_elements = test_municipality_soup.find_all(
    lambda tag: (
        tag.name in ["a", "input"]
        and (
            "WardDataPager" in str(tag)
            or "uxWardDataPager" in str(tag)
        )
    )
)

print(
    "uMdoni pagination controls found:",
    len(test_pager_elements)
)

assert len(test_pager_elements) > 0, (
    "uMdoni pagination controls were not found."
)

print("\nPagination controls:\n")

for element in test_pager_elements:
    print(
        "TAG:", element.name,
        "| TEXT:", element.get_text(" ", strip=True),
        "| NAME:", element.get("name"),
        "| VALUE:", element.get("value"),
        "| HREF:", element.get("href")
    )

uMdoni pagination controls found: 2

Pagination controls:

TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl00$ctl00 | VALUE: Previous | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl02$ctl00 | VALUE: Next | HREF: None


well this is good,  this tells us something important. uMdoni has only one page of wards. 

The controls are:

Previous → disabled/not a page number
Next → pagination exists, but there are no numbered page buttons

So we should not assume that numbered pages are always available. For uMdoni, the correct approach is to follow the Next control until it disappears/changes state.

Before automating all 44 municipalities, let's test that behavior once.

In [25]:
#25.
# uMdoni does not expose numbered page buttons.
# We therefore first extract the current page and
# determine how many ward records it contains.
#
# Pagination automation will be handled after
# this page is verified.

um_doni_rows = []

for row in test_ward_table.find_all("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["td", "th"])
    ]

    if len(cells) == 3 and cells[0].isdigit():
        um_doni_rows.append(cells)

print("uMdoni ward records on current page:", len(um_doni_rows))

assert len(um_doni_rows) > 0, (
    "No uMdoni ward records were extracted."
)

print("\nFirst 5 uMdoni records:")

for row in um_doni_rows[:5]:
    print(row)

print("\nuMdoni current-page extraction passed.")

uMdoni ward records on current page: 19

First 5 uMdoni records:
['52102001', '10', '3,992']
['52102002', '10', '4,556']
['52102003', '6', '5,082']
['52102004', '8', '4,008']
['52102005', '5', '4,713']

uMdoni current-page extraction passed.


Before we build the general paginator, let's verify that the Next control is actually disabled when there is no second page. This is important so our eventual loop knows when to stop.

In [26]:
#26.
# controls. We now inspect the Next control to
# determine whether another page is available.
#
# This will guide the dynamic pagination logic
# used for all KZN municipalities.

next_control = None

for element in test_pager_elements:
    if element.get("value", "").strip() == "Next":
        next_control = element
        break

assert next_control is not None, (
    "uMdoni Next pagination control was not found."
)

print("Next control name:")
print(next_control.get("name"))

print("\nNext control value:")
print(next_control.get("value"))

print("\nNext control HTML:")
print(next_control)

print("\nNext control inspection completed.")

Next control name:
ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl02$ctl00

Next control value:
Next

Next control HTML:
<input class="aspNetDisabled btn btn-link font-weight-bold" disabled="disabled" name="ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl02$ctl00" type="submit" value="Next"/>

Next control inspection completed.


So uMdoni's Next button exists in the HTML, but it is disabled, meaning there is no second page.

That gives us the rule we need for the general scraper:

Keep following pagination only while the Next control is enabled.

We now have verified both pagination cases:

eThekwini: numbered pages then 5 pages then 92 wards
uMdoni: no numbered pages then Next disabled then 19 wards

okay based on the logic we have established and test

let create a small function that extracts ward rows from any IEC response. This doesn't make requests yet; it just standardizes the extraction logic we've already tested.

In [27]:
#27.# This function extracts the IEC ward registration
# records from a municipality response.
#
# It only extracts the raw values returned by the IEC.
# No cleaning or transformation is performed.

def extract_ward_rows(response_text):
    """
    Extract ward-level registration rows from
    an IEC municipality AJAX response.
    """

    soup = BeautifulSoup(
        response_text,
        "html.parser"
    )

    ward_table = None

    for table in soup.find_all("table"):
        table_text = table.get_text(
            " ",
            strip=True
        )

        if (
            "Ward" in table_text
            and "Voting districts" in table_text
            and "Registered voters" in table_text
        ):
            ward_table = table
            break

    assert ward_table is not None, (
        "IEC ward registration table not found."
    )

    rows = []

    for row in ward_table.find_all("tr"):
        cells = [
            cell.get_text(" ", strip=True)
            for cell in row.find_all(
                ["td", "th"]
            )
        ]

        if len(cells) == 3 and cells[0].isdigit():
            rows.append(cells)

    assert len(rows) > 0, (
        "No ward records found."
    )

    return rows


# Test the function against the verified uMdoni response
test_rows = extract_ward_rows(
    test_municipality_response.text
)

print("Rows extracted:", len(test_rows))

print("\nFirst 5 rows:")
for row in test_rows[:5]:
    print(row)

assert test_rows == um_doni_rows

print("\nWard extraction function verification passed.")

Rows extracted: 19

First 5 rows:
['52102001', '10', '3,992']
['52102002', '10', '4,556']
['52102003', '6', '5,082']
['52102004', '8', '4,008']
['52102005', '5', '4,713']

Ward extraction function verification passed.


We now have a reusable ward extractor that has been tested against the real uMdoni response.

The next piece is the important one: dynamic pagination. We need a function that can handle both:

municipalities with numbered pages like eThekwini, and
municipalities where only Next/Previous controls are present.

to be safe maybe lets try inspecting the IEC voter registrating site NEXT than assuming how many ward pages it has per municipality

In [28]:
#28.

# The IEC uses different pagination layouts depending
# on the number of ward records.
#
# This helper checks whether the Next button is actually
# enabled. A disabled Next button means the current page
# is the final page.

def get_next_pager_control(response_text):
    """
    Find the IEC ward pager's Next control and
    determine whether another page is available.
    """

    soup = BeautifulSoup(
        response_text,
        "html.parser"
    )

    pager_elements = soup.find_all(
        lambda tag: (
            tag.name in ["a", "input"]
            and (
                "WardDataPager" in str(tag)
                or "uxWardDataPager" in str(tag)
            )
        )
    )

    for element in pager_elements:

        if element.get("value", "").strip() == "Next":

            is_disabled = (
                element.has_attr("disabled")
                or "aspNetDisabled" in (
                    element.get("class") or []
                )
            )

            return element, not is_disabled

        if element.get_text(" ", strip=True) == "Next":

            is_disabled = (
                element.has_attr("disabled")
                or "aspNetDisabled" in (
                    element.get("class") or []
                )
            )

            return element, not is_disabled

    return None, False


# Test against the verified uMdoni response
next_element, next_enabled = get_next_pager_control(
    test_municipality_response.text
)

print("Next control found:", next_element is not None)
print("Next enabled:", next_enabled)

assert next_element is not None
assert next_enabled is False

print("\nuMdoni final-page detection passed.")

Next control found: True
Next enabled: False

uMdoni final-page detection passed.


In [29]:
#29. okay before we move we need to be sure we getting right data while considering the number 4 of the 5vs mostly 
# lets use a function that dynamic for next poage testing

# eThekwini is used because we already verified that
# it has multiple ward pages.
#
# This test identifies the enabled Next control and
# sends the corresponding ASP.NET request.

# Check the Next control on the original eThekwini page
ethekwini_next_element, ethekwini_next_enabled = (
    get_next_pager_control(
        municipality_response.text
    )
)

print("eThekwini Next control found:",
      ethekwini_next_element is not None)

print("eThekwini Next enabled:",
      ethekwini_next_enabled)

assert ethekwini_next_element is not None
assert ethekwini_next_enabled is True

# Extract the control name used by ASP.NET
ethekwini_next_control = (
    ethekwini_next_element.get("name")
)

print("\nNext control:")
print(ethekwini_next_control)

assert ethekwini_next_control is not None

# Get the latest ASP.NET state from the current eThekwini page
ethekwini_current_state = {}

for name, value in re.findall(
    r"\|hiddenField\|([^|]+)\|([^|]*)",
    municipality_response.text
):
    ethekwini_current_state[name] = value

assert "__VIEWSTATE" in ethekwini_current_state
assert "__EVENTVALIDATION" in ethekwini_current_state

# Build the Next-page request
ethekwini_next_post_data = (
    ethekwini_current_state.copy()
)

ethekwini_next_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        + ethekwini_next_control,

    "__EVENTTARGET":
        ethekwini_next_control,

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "4005",

    "__ASYNCPOST":
        "true"
})

ethekwini_next_response = iec_session.post(
    IEC_URL,
    data=ethekwini_next_post_data,
    headers=province_headers,
    timeout=30
)

print("\nNext-page request:")
print("HTTP status:",
      ethekwini_next_response.status_code)

print(
    "Response size:",
    len(ethekwini_next_response.content),
    "bytes"
)

assert ethekwini_next_response.status_code == 200
assert "updatePanel|MainContent_MainUpdatePanel" in (
    ethekwini_next_response.text
)

# Extract the resulting ward page
ethekwini_next_rows = extract_ward_rows(
    ethekwini_next_response.text
)

print(
    "Ward records returned:",
    len(ethekwini_next_rows)
)

print("\nFirst 5 records:")
for row in ethekwini_next_rows[:5]:
    print(row)

print("\nDynamic Next-page request passed.")

eThekwini Next control found: True
eThekwini Next enabled: True

Next control:
ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl02$ctl00

Next-page request:
HTTP status: 200
Response size: 84082 bytes
Ward records returned: 20

First 5 records:
['59500021', '6', '18,025']
['59500022', '6', '16,753']
['59500023', '6', '21,174']
['59500024', '6', '17,529']
['59500025', '6', '18,854']

Dynamic Next-page request passed.


This is the key test we needed. We can now move through pages using the actual enabled Next control instead of hard-coding page numbers.

So the scraper can handle:

eThekwini: multiple pages → follows Next
uMdoni: one page → Next disabled, stops
Final pages: can contain fewer than 20 wards
ASP.NET state is refreshed between requests

so now we can move on and build the dynamic municipality paginator

Now we'll turn that tested logic into one reusable function. This cell creates the function and tests it on eThekwini. It does not touch all 44 municipalities yet.

In [30]:
#30.
# This function extracts all ward pages for one
# municipality by repeatedly following the IEC's
# enabled Next control.
#
# It does not assume:
# - a fixed number of pages
# - 20 records per page
# - numbered page controls
#
# The function stops when the IEC Next control
# becomes disabled.

def extract_municipality_wards(
    municipality_id,
    first_response_text
):
    """
    Extract all ward records for one municipality
    from the IEC's ASP.NET AJAX pagination system.
    """

    all_rows = []
    current_response_text = first_response_text
    page_number = 1

    while True:

        # Extract current page's ward records
        current_rows = extract_ward_rows(
            current_response_text
        )

        all_rows.extend(current_rows)

        print(
            f"Page {page_number}: "
            f"{len(current_rows)} wards"
        )

        # Check whether another page exists
        next_element, next_enabled = (
            get_next_pager_control(
                current_response_text
            )
        )

        if not next_enabled:
            break

        next_control = next_element.get("name")

        assert next_control is not None, (
            "Enabled Next control has no name."
        )

        # Extract latest ASP.NET state
        current_state = {}

        for name, value in re.findall(
            r"\|hiddenField\|([^|]+)\|([^|]*)",
            current_response_text
        ):
            current_state[name] = value

        assert "__VIEWSTATE" in current_state
        assert "__EVENTVALIDATION" in current_state

        # Build next-page request
        next_post_data = current_state.copy()

        next_post_data.update({
            "ctl00$ctl13":
                "ctl00$MainContent$MainUpdatePanel|"
                + next_control,

            "__EVENTTARGET":
                next_control,

            "__EVENTARGUMENT": "",
            "__LASTFOCUS": "",

            "ctl00$MainContent$ddlProvinces":
                KZN_PROVINCE_ID,

            "ctl00$MainContent$ddlMunicipalities":
                municipality_id,

            "__ASYNCPOST":
                "true"
        })

        next_response = iec_session.post(
            IEC_URL,
            data=next_post_data,
            headers=province_headers,
            timeout=30
        )

        assert next_response.status_code == 200
        assert (
            "updatePanel|MainContent_MainUpdatePanel"
            in next_response.text
        )

        current_response_text = (
            next_response.text
        )

        page_number += 1

    return all_rows


# Test the reusable paginator on eThekwini
ethekwini_test_rows = extract_municipality_wards(
    "4005",
    municipality_response.text
)

print("\nTotal eThekwini wards extracted:",
      len(ethekwini_test_rows))

assert len(ethekwini_test_rows) == 92

assert len({
    row[0]
    for row in ethekwini_test_rows
}) == 92

print(
    "\nDynamic eThekwini pagination "
    "verification passed."
)

Page 1: 20 wards
Page 2: 20 wards
Page 3: 20 wards
Page 4: 20 wards
Page 5: 20 wards
Page 6: 12 wards

Total eThekwini wards extracted: 112


AssertionError: 

In [ ]:
#31. eish did not work we back to diagnose then lets diagnose the paginator output

# We inspect the actual number of records and
# duplicates before changing the extraction logic.

print(
    "Total records returned:",
    len(ethekwini_test_rows)
)

ethekwini_test_wards = [
    row[0]
    for row in ethekwini_test_rows
]

print(
    "Unique ward IDs:",
    len(set(ethekwini_test_wards))
)

print(
    "Duplicate records:",
    len(ethekwini_test_wards)
    - len(set(ethekwini_test_wards))
)

print("\nFirst 5 records:")
for row in ethekwini_test_rows[:5]:
    print(row)

print("\nLast 5 records:")
for row in ethekwini_test_rows[-5:]:
    print(row)

print("\nExpected records: 92")

Let's modify the test slightly so we can see the pagination controls on the response that produced the extra records.

In [31]:
#32. The paginator returned 112 records instead of 92.
# We therefore inspect the pagination controls after
# the fifth page to see what the IEC is exposing as
# the next action.

# Start again from the original eThekwini response
debug_response_text = municipality_response.text

for page_number in range(1, 6):

    next_element, next_enabled = (
        get_next_pager_control(
            debug_response_text
        )
    )

    print(
        f"Page {page_number} -> "
        f"Next found: {next_element is not None}, "
        f"Next enabled: {next_enabled}"
    )

    if not next_enabled:
        break

    next_control = next_element.get("name")

    debug_state = {}

    for name, value in re.findall(
        r"\|hiddenField\|([^|]+)\|([^|]*)",
        debug_response_text
    ):
        debug_state[name] = value

    debug_post_data = debug_state.copy()

    debug_post_data.update({
        "ctl00$ctl13":
            "ctl00$MainContent$MainUpdatePanel|"
            + next_control,

        "__EVENTTARGET": next_control,
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": "",

        "ctl00$MainContent$ddlProvinces":
            KZN_PROVINCE_ID,

        "ctl00$MainContent$ddlMunicipalities":
            "4005",

        "__ASYNCPOST": "true"
    })

    debug_response = iec_session.post(
        IEC_URL,
        data=debug_post_data,
        headers=province_headers,
        timeout=30
    )

    assert debug_response.status_code == 200

    debug_response_text = debug_response.text

print("\nPagination controls on the fifth-page response:")

debug_soup = BeautifulSoup(
    debug_response_text,
    "html.parser"
)

debug_pager_elements = debug_soup.find_all(
    lambda tag: (
        tag.name in ["a", "input"]
        and (
            "WardDataPager" in str(tag)
            or "uxWardDataPager" in str(tag)
        )
    )
)

for element in debug_pager_elements:
    print(
        "TAG:", element.name,
        "| VALUE:", element.get("value"),
        "| NAME:", element.get("name"),
        "| DISABLED:", element.has_attr("disabled"),
        "| CLASS:", element.get("class")
    )

Page 1 -> Next found: True, Next enabled: True
Page 2 -> Next found: True, Next enabled: True
Page 3 -> Next found: True, Next enabled: True
Page 4 -> Next found: True, Next enabled: True
Page 5 -> Next found: True, Next enabled: True

Pagination controls on the fifth-page response:
TAG: input | VALUE: Previous | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl00$ctl00 | DISABLED: False | CLASS: ['btn', 'btn-link', 'font-weight-bold']
TAG: input | VALUE: ... | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl00 | DISABLED: False | CLASS: None
TAG: input | VALUE: Next | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl02$ctl00 | DISABLED: True | CLASS: ['aspNetDisabled', 'btn', 'btn-link', 'font-weight-bold']


In [32]:
#33. eThekwini exposes numbered page controls for its
# five-page result set.
#
# We inspect the controls returned on the first page
# so we can identify the actual page-navigation pattern
# before building the final paginator.

ethekwini_first_page_pager = municipality_soup.find_all(
    lambda tag: (
        tag.name in ["a", "input"]
        and (
            "WardDataPager" in str(tag)
            or "uxWardDataPager" in str(tag)
        )
    )
)

print("eThekwini pager controls:")

for element in ethekwini_first_page_pager:
    print(
        "VALUE:", element.get("value"),
        "| NAME:", element.get("name"),
        "| DISABLED:", element.has_attr("disabled"),
        "| CLASS:", element.get("class")
    )

eThekwini pager controls:
VALUE: Previous | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl00$ctl00 | DISABLED: True | CLASS: ['aspNetDisabled', 'btn', 'btn-link', 'font-weight-bold']
VALUE: 2 | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl01 | DISABLED: False | CLASS: ['btn', 'btn-link', 'font-weight-bold']
VALUE: 3 | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl02 | DISABLED: False | CLASS: ['btn', 'btn-link', 'font-weight-bold']
VALUE: 4 | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl03 | DISABLED: False | CLASS: ['btn', 'btn-link', 'font-weight-bold']
VALUE: 5 | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl04 | DISABLED: False | CLASS: ['btn', 'btn-link', 'font-weight-bold']
VALUE: ... | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl05 | DISABLED: False | CLASS: None
VALUE: Next | NAM

In [33]:
#34.

# The IEC pager can expose numbered page buttons,
# such as 2, 3, 4 and 5.
#
# This helper identifies those controls while ignoring:
# - Previous
# - Next
# - ...
#
# We use the actual control names returned by the IEC.

def get_numbered_page_controls(response_text):
    """
    Return numbered IEC pager controls as:
    [(page_number, control_name), ...]
    """

    soup = BeautifulSoup(
        response_text,
        "html.parser"
    )

    controls = []

    pager_elements = soup.find_all(
        lambda tag: (
            tag.name in ["a", "input"]
            and (
                "WardDataPager" in str(tag)
                or "uxWardDataPager" in str(tag)
            )
        )
    )

    for element in pager_elements:

        value = element.get("value", "").strip()

        if not value:
            value = element.get_text(
                " ",
                strip=True
            )

        if value.isdigit():

            controls.append({
                "page": int(value),
                "name": element.get("name"),
                "disabled": element.has_attr("disabled")
            })

    return controls


# Test against eThekwini Page 1
ethekwini_numbered_pages = (
    get_numbered_page_controls(
        municipality_response.text
    )
)

print("Numbered page controls found:",
      len(ethekwini_numbered_pages))

for control in ethekwini_numbered_pages:
    print(control)

assert [control["page"]
        for control in ethekwini_numbered_pages] == [
            2, 3, 4, 5
        ]

print("\nNumbered page control detection passed.")

Numbered page controls found: 4
{'page': 2, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl01', 'disabled': False}
{'page': 3, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl02', 'disabled': False}
{'page': 4, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl03', 'disabled': False}
{'page': 5, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl04', 'disabled': False}

Numbered page control detection passed.


In [34]:
#35.
# We use the detected Page 2 control rather than
# hard-coding its ASP.NET control name.
#
# This confirms that the numbered-page helper can
# drive the IEC pagination correctly.

page2_info = next(
    control
    for control in ethekwini_numbered_pages
    if control["page"] == 2
)

page2_control = page2_info["name"]

print("Page requested:", page2_info["page"])
print("Control:", page2_control)

# Extract current ASP.NET state
page2_test_state = {}

for name, value in re.findall(
    r"\|hiddenField\|([^|]+)\|([^|]*)",
    municipality_response.text
):
    page2_test_state[name] = value

assert "__VIEWSTATE" in page2_test_state
assert "__EVENTVALIDATION" in page2_test_state

page2_test_data = page2_test_state.copy()

page2_test_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        + page2_control,

    "__EVENTTARGET":
        page2_control,

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "4005",

    "__ASYNCPOST":
        "true"
})

page2_test_response = iec_session.post(
    IEC_URL,
    data=page2_test_data,
    headers=province_headers,
    timeout=30
)

print("\nHTTP status:",
      page2_test_response.status_code)

print(
    "Response size:",
    len(page2_test_response.content),
    "bytes"
)

assert page2_test_response.status_code == 200

page2_test_rows = extract_ward_rows(
    page2_test_response.text
)

print(
    "Page 2 ward records:",
    len(page2_test_rows)
)

print("\nFirst 5 records:")
for row in page2_test_rows[:5]:
    print(row)

assert page2_test_rows == page2_rows

print("\nNumbered Page 2 request verification passed.")

Page requested: 2
Control: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl01

HTTP status: 200
Response size: 84082 bytes
Page 2 ward records: 20

First 5 records:
['59500021', '6', '18,025']
['59500022', '6', '16,753']
['59500023', '6', '21,174']
['59500024', '6', '17,529']
['59500025', '6', '18,854']


AssertionError: 

In [35]:
#36. We verify the numbered-page request using the
# actual ward IDs rather than comparing two lists
# created through different extraction paths.

expected_page2_wards = {
    row[0]
    for row in page2_rows
}

actual_page2_wards = {
    row[0]
    for row in page2_test_rows
}

print("Expected Page 2 wards:", len(expected_page2_wards))
print("Actual Page 2 wards:", len(actual_page2_wards))

print(
    "Ward IDs match:",
    expected_page2_wards == actual_page2_wards
)

assert len(page2_test_rows) == 20
assert len(actual_page2_wards) == 20
assert actual_page2_wards == expected_page2_wards

print("\nNumbered Page 2 verification passed.")

Expected Page 2 wards: 20
Actual Page 2 wards: 20
Ward IDs match: False


AssertionError: 

In [36]:
#37
# The numbered Page 2 request returned 20 records,
# but its ward IDs did not match the previously stored
# page2_rows exactly.
#
# We inspect the differences before changing any
# pagination logic.

expected_page2_wards = {
    row[0]
    for row in page2_rows
}

actual_page2_wards = {
    row[0]
    for row in page2_test_rows
}

only_in_expected = sorted(
    expected_page2_wards - actual_page2_wards
)

only_in_actual = sorted(
    actual_page2_wards - expected_page2_wards
)

print("Expected Page 2 ward IDs:")
print(sorted(expected_page2_wards))

print("\nActual Page 2 ward IDs:")
print(sorted(actual_page2_wards))

print("\nWard IDs only in expected set:")
print(only_in_expected)

print("\nWard IDs only in actual set:")
print(only_in_actual)

print("\nExpected count:", len(expected_page2_wards))
print("Actual count:", len(actual_page2_wards))

Expected Page 2 ward IDs:
['59500041', '59500042', '59500043', '59500044', '59500045', '59500046', '59500047', '59500048', '59500049', '59500050', '59500051', '59500052', '59500053', '59500054', '59500055', '59500056', '59500057', '59500058', '59500059', '59500060']

Actual Page 2 ward IDs:
['59500021', '59500022', '59500023', '59500024', '59500025', '59500026', '59500027', '59500028', '59500029', '59500030', '59500031', '59500032', '59500033', '59500034', '59500035', '59500036', '59500037', '59500038', '59500039', '59500040']

Ward IDs only in expected set:
['59500041', '59500042', '59500043', '59500044', '59500045', '59500046', '59500047', '59500048', '59500049', '59500050', '59500051', '59500052', '59500053', '59500054', '59500055', '59500056', '59500057', '59500058', '59500059', '59500060']

Ward IDs only in actual set:
['59500021', '59500022', '59500023', '59500024', '59500025', '59500026', '59500027', '59500028', '59500029', '59500030', '59500031', '59500032', '59500033', '595000

In [38]:
#38.  lets verufy page 3

page3_controls = get_numbered_page_controls(
    municipality_response.text
)

print("Numbered page controls found:")
for control in page3_controls:
    print(control)

# Find the control for Page 3
page3_control = next(
    control for control in page3_controls
    if control["page"] == 3
)

print("\nPage 3 control:")
print(page3_control)

assert page3_control["name"] == (
    "ctl00$MainContent$uxRegisteredVotersWardListView"
    "$uxWardDataPager$ctl01$ctl02"
)

print("\nPAGE 3 CONTROL VERIFICATION: PASSED")

Numbered page controls found:
{'page': 2, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl01', 'disabled': False}
{'page': 3, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl02', 'disabled': False}
{'page': 4, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl03', 'disabled': False}
{'page': 5, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl04', 'disabled': False}

Page 3 control:
{'page': 3, 'name': 'ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl02', 'disabled': False}

PAGE 3 CONTROL VERIFICATION: PASSED


In [41]:
#39. 
# ============================================
# CELL 40 — REBUILD IEC SESSION STATE
# ============================================

# Start a fresh IEC session so all ASP.NET state
# is created consistently in one place.

session = requests.Session()

# Browser-like headers required by the IEC ASP.NET page.
base_headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "X-Requested-With": "XMLHttpRequest",
    "X-MicrosoftAjax": "Delta=true",
    "Referer": IEC_URL,
    "Origin": "https://www.elections.org.za"
}

# --------------------------------------------
# 1. Load the IEC page
# --------------------------------------------

initial_response = session.get(
    IEC_URL,
    headers=base_headers,
    timeout=30
)

print("Initial page status:", initial_response.status_code)

assert initial_response.status_code == 200


# --------------------------------------------
# 2. Extract ASP.NET hidden state
# --------------------------------------------

initial_soup = BeautifulSoup(
    initial_response.text,
    "html.parser"
)

aspnet_fields = {}

for field_name in [
    "__VIEWSTATE",
    "__VIEWSTATEGENERATOR",
    "__EVENTVALIDATION"
]:
    field = initial_soup.find(
        "input",
        {"name": field_name}
    )

    if field:
        aspnet_fields[field_name] = field.get(
            "value",
            ""
        )

assert len(aspnet_fields) == 3

print("ASP.NET state fields:", list(aspnet_fields.keys()))


# --------------------------------------------
# 3. Select KwaZulu-Natal
# --------------------------------------------

province_post_data = aspnet_fields.copy()

province_post_data.update({
    "ctl00$MainContent$ddlProvinces": KZN_PROVINCE_ID,
    "ctl00$MainContent$ddlMunicipalities": "-1",

    "ctl00$MainContent$MainUpdatePanel":
        "ctl00$MainContent$ddlProvinces",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "__EVENTTARGET":
        "ctl00$MainContent$ddlProvinces",

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",
    "__SCROLLPOSITIONX": "0",
    "__SCROLLPOSITIONY": "0",

    "__ASYNCPOST": "true"
})

province_response = session.post(
    IEC_URL,
    data=province_post_data,
    headers=base_headers,
    timeout=30
)

print("Province request status:", province_response.status_code)
print("Province response size:", len(province_response.text))

assert province_response.status_code == 200


# --------------------------------------------
# 4. Extract updated ASP.NET state
# --------------------------------------------

province_soup = BeautifulSoup(
    province_response.text,
    "html.parser"
)

province_form_state = {}

for field_name in [
    "__VIEWSTATE",
    "__VIEWSTATEGENERATOR",
    "__EVENTVALIDATION"
]:
    field = province_soup.find(
        "input",
        {"name": field_name}
    )

    if field:
        province_form_state[field_name] = field.get(
            "value",
            ""
        )

assert len(province_form_state) == 3

print("Updated province state captured.")

print("\nIEC SESSION STATE: READY")

Initial page status: 200
ASP.NET state fields: ['__VIEWSTATE', '__VIEWSTATEGENERATOR', '__EVENTVALIDATION']
Province request status: 200
Province response size: 129533


AssertionError: 

In [42]:
#40.

def extract_aspnet_state(response_text):
    """
    Extract ASP.NET hidden form fields from an
    IEC HTML/AJAX response.
    """

    state = {}

    # First try normal HTML inputs.
    soup = BeautifulSoup(
        response_text,
        "html.parser"
    )

    for field_name in [
        "__VIEWSTATE",
        "__VIEWSTATEGENERATOR",
        "__EVENTVALIDATION"
    ]:
        field = soup.find(
            "input",
            {"name": field_name}
        )

        if field:
            state[field_name] = field.get(
                "value",
                ""
            )

    # If the response is an ASP.NET AJAX delta,
    # the hidden fields may appear as:
    #
    # hiddenField|__VIEWSTATE|VALUE
    #
    # rather than normal <input> elements.
    for field_name in [
        "__VIEWSTATE",
        "__VIEWSTATEGENERATOR",
        "__EVENTVALIDATION"
    ]:

        if field_name not in state:

            marker = f"hiddenField|{field_name}|"

            start = response_text.find(marker)

            if start != -1:

                value_start = start + len(marker)

                value_end = response_text.find(
                    "|",
                    value_start
                )

                if value_end == -1:
                    value_end = len(response_text)

                state[field_name] = response_text[
                    value_start:value_end
                ]

    return state


province_form_state = extract_aspnet_state(
    province_response.text
)

print(
    "State fields found:",
    list(province_form_state.keys())
)

for field_name, value in province_form_state.items():
    print(
        field_name,
        "length:",
        len(value)
    )

assert "__VIEWSTATE" in province_form_state
assert "__VIEWSTATEGENERATOR" in province_form_state
assert "__EVENTVALIDATION" in province_form_state

print("\nIEC STATE EXTRACTION: PASSED")

State fields found: ['__VIEWSTATE', '__VIEWSTATEGENERATOR', '__EVENTVALIDATION']
__VIEWSTATE length: 14320
__VIEWSTATEGENERATOR length: 8
__EVENTVALIDATION length: 2268

IEC STATE EXTRACTION: PASSED


In [43]:
# ============================================
# CELL 42 — SELECT TEST MUNICIPALITY
# ============================================

TEST_MUNICIPALITY_ID = "4005"
TEST_MUNICIPALITY_NAME = "ETH - eThekwini"

municipality_post_data = province_form_state.copy()

municipality_post_data.update({
    "ctl00$MainContent$ddlMunicipalities":
        TEST_MUNICIPALITY_ID,

    "__EVENTTARGET":
        "ctl00$MainContent$ddlMunicipalities",

    "__EVENTARGUMENT": "",
    "__ASYNCPOST": "true"
})

municipality_response = session.post(
    IEC_URL,
    data=municipality_post_data,
    headers=base_headers,
    timeout=30
)

print("HTTP status:", municipality_response.status_code)
print("Response size:", len(municipality_response.text))

assert municipality_response.status_code == 200

test_rows = extract_ward_rows(
    municipality_response.text
)

print("Municipality:", TEST_MUNICIPALITY_NAME)
print("First page rows:", len(test_rows))
print("First ward:", test_rows[0])
print("Last ward:", test_rows[-1])

assert len(test_rows) > 0

print("\nMUNICIPALITY SELECTION: PASSED")

HTTP status: 200
Response size: 84122
Municipality: ETH - eThekwini
First page rows: 20
First ward: ['59500001', '14', '19,964']
Last ward: ['59500020', '7', '16,140']

MUNICIPALITY SELECTION: PASSED


In [44]:
# 43. lets try and autoimate the extraction now:

def extract_all_ward_pages(first_response_text):
    """
    Extract all ward records by following the IEC
    'Next' pagination control until it is disabled.
    """

    all_rows = []
    current_response_text = first_response_text
    page_number = 1

    while True:

        # Extract current page
        current_rows = extract_ward_rows(
            current_response_text
        )

        all_rows.extend(current_rows)

        print(
            f"Page {page_number}: "
            f"{len(current_rows)} wards"
        )

        # Find the Next button
        next_element, next_enabled = (
            get_next_pager_control(
                current_response_text
            )
        )

        # Stop when Next is disabled
        if not next_enabled:
            break

        next_control_name = next_element.get("name")

        # Build request from the current page's
        # ASP.NET state.
        current_state = extract_aspnet_state(
            current_response_text
        )

        next_post_data = current_state.copy()

        next_post_data.update({
            next_control_name: "Next",
            "__EVENTTARGET": next_control_name,
            "__EVENTARGUMENT": "",
            "__ASYNCPOST": "true"
        })

        next_response = session.post(
            IEC_URL,
            data=next_post_data,
            headers=base_headers,
            timeout=30
        )

        assert next_response.status_code == 200

        current_response_text = (
            next_response.text
        )

        page_number += 1

    return all_rows


# Run the paginator on eThekwini
ethekwini_all_rows = extract_all_ward_pages(
    municipality_response.text
)

print("\nTotal wards extracted:", len(ethekwini_all_rows))

# Basic final verification
ward_ids = [
    row[0]
    for row in ethekwini_all_rows
]

print(
    "Unique ward IDs:",
    len(set(ward_ids))
)

print(
    "First ward:",
    ethekwini_all_rows[0]
)

print(
    "Last ward:",
    ethekwini_all_rows[-1]
)

assert len(ethekwini_all_rows) == len(set(ward_ids))
assert len(ethekwini_all_rows) > 20

print("\nAUTOMATIC PAGINATION: PASSED")

Page 1: 20 wards
Page 2: 20 wards
Page 3: 20 wards
Page 4: 20 wards
Page 5: 20 wards
Page 6: 12 wards

Total wards extracted: 112
Unique ward IDs: 112
First ward: ['59500001', '14', '19,964']
Last ward: ['59500112', '6', '17,140']

AUTOMATIC PAGINATION: PASSED


In [45]:
#43. 

ethekwini_df = pd.DataFrame(
    ethekwini_all_rows,
    columns=[
        "ward",
        "voting_districts",
        "registered_voters"
    ]
)

# Keep the data exactly as extracted from the IEC.
# No cleaning or transformation is performed here.

output_file = (
    RAW_DIR
    / "iec_kzn_ethekwini_ward_registration_2026.csv"
)

ethekwini_df.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Rows:", len(ethekwini_df))
print("Columns:", list(ethekwini_df.columns))

# Verify the saved raw file can be read back.
saved_check = pd.read_csv(output_file)

assert len(saved_check) == 112
assert list(saved_check.columns) == [
    "ward",
    "voting_districts",
    "registered_voters"
]

print("\nSaved file verification: PASSED")

Saved: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw\iec_kzn_ethekwini_ward_registration_2026.csv
Rows: 112
Columns: ['ward', 'voting_districts', 'registered_voters']

Saved file verification: PASSED


In [48]:
#44. lest test this logic one more time so lets do that as the next single cell, 
#then test it on uMdoni (which we already know has 19 wards) 
#before letting it run across all 43 remaining municipalities
# if it gives us 19 wards we will then let it extract therest of the municplaities
#it would mean its good to go


def extract_municipality_wards(
    municipality_id,
    municipality_name,
    province_state
):
    """
    Extract all ward-level voter registration records
    for one KZN municipality.

    The IEC's Next pagination control is followed
    automatically until the final page is reached.
    """

    # ----------------------------------------
    # Select municipality
    # ----------------------------------------

    municipality_post_data = province_state.copy()

    municipality_post_data.update({
        "ctl00$MainContent$ddlMunicipalities":
            municipality_id,

        "__EVENTTARGET":
            "ctl00$MainContent$ddlMunicipalities",

        "__EVENTARGUMENT": "",
        "__ASYNCPOST": "true"
    })

    municipality_response = session.post(
        IEC_URL,
        data=municipality_post_data,
        headers=base_headers,
        timeout=30
    )

    assert municipality_response.status_code == 200

    # ----------------------------------------
    # Extract all pages
    # ----------------------------------------

    all_rows = []
    current_response_text = municipality_response.text
    page_number = 1

    while True:

        current_rows = extract_ward_rows(
            current_response_text
        )

        all_rows.extend(current_rows)

        print(
            f"{municipality_name} | "
            f"Page {page_number}: "
            f"{len(current_rows)} wards"
        )

        # Find Next
        next_element, next_enabled = (
            get_next_pager_control(
                current_response_text
            )
        )

        if not next_enabled:
            break

        next_control_name = next_element.get("name")

        # Get the ASP.NET state belonging to
        # the current page.
        current_state = extract_aspnet_state(
            current_response_text
        )

        next_post_data = current_state.copy()

        next_post_data.update({
            next_control_name: "Next",
            "__EVENTTARGET": next_control_name,
            "__EVENTARGUMENT": "",
            "__ASYNCPOST": "true"
        })

        next_response = session.post(
            IEC_URL,
            data=next_post_data,
            headers=base_headers,
            timeout=30
        )

        assert next_response.status_code == 200

        current_response_text = (
            next_response.text
        )

        page_number += 1

    # ----------------------------------------
    # Final verification
    # ----------------------------------------

    ward_ids = [
        row[0]
        for row in all_rows
    ]

    assert len(all_rows) > 0
    assert len(ward_ids) == len(set(ward_ids))

    print(
        f"{municipality_name} | "
        f"Total wards: {len(all_rows)}"
    )

    return all_rows





In [49]:

UMDONI_ID = "4403"
UMDONI_NAME = "KZN212 - uMdoni"

umdonI_rows = extract_municipality_wards(
    municipality_id=UMDONI_ID,
    municipality_name=UMDONI_NAME,
    province_state=province_form_state
)

print("\nTEST RESULT")
print("Municipality:", UMDONI_NAME)
print("Total wards extracted:", len(umdonI_rows))
print("Unique wards:", len(set(row[0] for row in umdonI_rows)))

print("\nFirst 5 wards:")
for row in umdonI_rows[:5]:
    print(row)

assert len(umdonI_rows) == 19
assert len(set(row[0] for row in umdonI_rows)) == 19

print("\nUMDONI EXTRACTION TEST: PASSED")

KZN212 - uMdoni | Page 1: 19 wards
KZN212 - uMdoni | Total wards: 19

TEST RESULT
Municipality: KZN212 - uMdoni
Total wards extracted: 19
Unique wards: 19

First 5 wards:
['52102001', '10', '3,992']
['52102002', '10', '4,556']
['52102003', '6', '5,082']
['52102004', '8', '4,008']
['52102005', '5', '4,713']

UMDONI EXTRACTION TEST: PASSED


In [50]:
#45 

# eThekwini was already extracted and saved.
# uMdoni was used as the second test municipality.
#
# This cell extracts the remaining KZN municipalities
# and saves each municipality as its own raw CSV.
#
# No cleaning, transformation, or combining is performed.


# Get the KZN municipality list


kzn_municipalities = []

province_soup = BeautifulSoup(
    province_response.text,
    "html.parser"
)

municipality_select = province_soup.find(
    "select",
    {"id": "ctl00_MainContent_ddlMunicipalities"}
)

if municipality_select is None:
    # ASP.NET AJAX responses can sometimes require
    # searching by the select name instead.
    municipality_select = province_soup.find(
        "select",
        {"name": "ctl00$MainContent$ddlMunicipalities"}
    )

assert municipality_select is not None, (
    "KZN municipality dropdown was not found."
)

for option in municipality_select.find_all("option"):

    municipality_id = option.get("value", "").strip()
    municipality_name = option.get_text(
        " ",
        strip=True
    )

    if (
        municipality_id
        and municipality_id != "-1"
        and municipality_name
    ):
        kzn_municipalities.append({
            "id": municipality_id,
            "name": municipality_name
        })

print(
    "KZN municipalities found:",
    len(kzn_municipalities)
)

for municipality in kzn_municipalities:
    print(
        municipality["id"],
        "|",
        municipality["name"]
    )

assert len(kzn_municipalities) == 44

print("\nKZN MUNICIPALITY LIST: PASSED")



# Extract remaining municipalities


extraction_summary = []

for index, municipality in enumerate(
    kzn_municipalities,
    start=1
):

    municipality_id = municipality["id"]
    municipality_name = municipality["name"]

    # Skip eThekwini because it was already
    # extracted and saved.
    if municipality_id == TEST_MUNICIPALITY_ID:
        print(
            f"\n[{index}/44] "
            f"{municipality_name} — SKIPPED "
            f"(already extracted)"
        )
        continue

    print(
        f"\n{'=' * 60}"
    )

    print(
        f"[{index}/44] Extracting: "
        f"{municipality_name}"
    )

    print(
        f"Municipality ID: {municipality_id}"
    )

    try:

        rows = extract_municipality_wards(
            municipality_id=municipality_id,
            municipality_name=municipality_name,
            province_state=province_form_state
        )


        # Verification
 

        ward_ids = [
            row[0]
            for row in rows
        ]

        unique_ward_ids = set(ward_ids)

        assert len(rows) > 0
        assert len(rows) == len(unique_ward_ids)

  
        # Create raw dataframe
     

        municipality_df = pd.DataFrame(
            rows,
            columns=[
                "ward",
                "voting_districts",
                "registered_voters"
            ]
        )

   
        # Safe filename
   

        safe_name = (
            municipality_name
            .lower()
            .replace(" ", "_")
            .replace("/", "_")
            .replace("\\", "_")
            .replace("-", "_")
        )

        output_file = (
            RAW_DIR
            / f"iec_kzn_{safe_name}"
            "_ward_registration_2026.csv"
        )

        municipality_df.to_csv(
            output_file,
            index=False
        )

   
        # Verify saved file


        saved_df = pd.read_csv(
            output_file
        )

        assert len(saved_df) == len(rows)

        extraction_summary.append({
            "municipality_id": municipality_id,
            "municipality": municipality_name,
            "wards": len(rows),
            "file": output_file.name,
            "status": "SUCCESS"
        })

        print(
            f"✓ Saved {len(rows)} wards → "
            f"{output_file.name}"
        )

    except Exception as error:

        extraction_summary.append({
            "municipality_id": municipality_id,
            "municipality": municipality_name,
            "wards": 0,
            "file": "",
            "status": f"FAILED: {error}"
        })

        print(
            f"✗ FAILED: {municipality_name}"
        )

        print(
            "Error:",
            error
        )



# Final extraction summary


summary_df = pd.DataFrame(
    extraction_summary
)

print("\n")
print("=" * 60)
print("KZN EXTRACTION SUMMARY")
print("=" * 60)

print(
    summary_df[
        [
            "municipality",
            "wards",
            "status"
        ]
    ].to_string(index=False)
)

print(
    "\nSuccessful:",
    (
        summary_df["status"]
        == "SUCCESS"
    ).sum()
)

print(
    "Failed:",
    (
        summary_df["status"]
        .str.startswith("FAILED")
    ).sum()
)

KZN municipalities found: 44
4005 | ETH - eThekwini
4403 | KZN212 - uMdoni
4404 | KZN213 - uMzumbe
4405 | KZN214 - uMuziwabantu
4407 | KZN216 - Ray Nkonyeni
4409 | KZN221 - uMshwathi
4410 | KZN222 - uMngeni
4411 | KZN223 - Mpofana
4412 | KZN224 - iMpendle
4413 | KZN225 - Msunduzi
4414 | KZN226 - Mkhambathini
4415 | KZN227 - Richmond
4421 | KZN235 - Okhahlamba
4461 | KZN237 - iNkosi Langalibalele
4462 | KZN238 - Alfred Duma
4426 | KZN241 - eNdumeni
4427 | KZN242 - Nqutu
4429 | KZN244 - uMsinga
4430 | KZN245 - uMvoti
4432 | KZN252 - Newcastle
4433 | KZN253 - eMadlangeni
4434 | KZN254 - Dannhauser
4436 | KZN261 - eDumbe
4437 | KZN262 - uPhongolo
4438 | KZN263 - AbaQulusi
4439 | KZN265 - Nongoma
4440 | KZN266 - Ulundi
4442 | KZN271 - uMhlabuyalingana
4443 | KZN272 - Jozini
4446 | KZN275 - Inkosi uMtubatuba
4463 | KZN276 - Big Five Hlabisa
4449 | KZN281 - uMfolozi
4450 | KZN282 - uMhlathuze
4452 | KZN284 - uMlalazi
4453 | KZN285 - Mthonjaneni
4454 | KZN286 - Nkandla
4456 | KZN291 - Mandeni
